# Planner model performance

Performance metrics for the planner model that parse semantic searches.

In [ ]:
import sys
from pathlib import Path

ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "backend/app/configs/config.yaml").is_file()
)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
import matplotlib.pyplot as plt
from analyses.helpers.planner import (
    describe,
    load_planner_run,
    model_legend_handles,
    per_case_data,
    per_case_panel,
    quality_cost_panel,
    trade_off_data,
    unavailable_models,
)
from analyses.helpers.publication import export_figure, load_settings, publication_style

settings = load_settings(ROOT)
publication_style()

## Load one benchmark run

The manifest contains aggregate model metrics; `trials.jsonl` contains one record per model, case, and repeat. Absolute artifact paths recorded on another machine are intentionally ignored—the trials file is resolved beside the selected manifest.

In [ ]:
run = load_planner_run(ROOT)
print(describe(run))

## Prepare accuracy and cost data

Models without successful calls are flagged as provider failures and excluded from the quality–cost panels. Each panel highlights models for which no other model is at least as accurate and no more costly, with a strict improvement in one dimension. The per-case heatmap includes every model.

In [ ]:
unavailable = unavailable_models(run)

if unavailable:
    print(
        "WARNING: no successful calls for: "
        + ", ".join(unavailable)
        + ". Treat these as access/provider failures, not model-performance results."
    )

trade_off_data(run)

## Combined quality–cost and per-case accuracy

The top row shows the latency and token trade-offs. The bottom row shows the share of correct repeats for each benchmark case.

In [ ]:
fig_width = max(12, len(run.manifest["case_ids"]) * 0.9)
fig_height = max(6, len(run.models) * 0.7)
fig = plt.figure(figsize=(max(16, fig_width), max(10, 5.5 + fig_height)), layout="constrained")
top, bottom = fig.subfigures(2, 1, height_ratios=(5.5, fig_height))
top_axes = top.subplots(1, 3, gridspec_kw={"width_ratios": (1, 1, 0.55)})

for ax, (cost_column, xlabel, title) in zip(
    top_axes[:2],
    (
        ("latency_p50_seconds", "p50 latency (seconds)", "A) Accuracy vs. latency"),
        ("tokens_per_success", "Tokens per successful call", "B) Accuracy vs. token usage"),
    ),
    strict=True,
):
    quality_cost_panel(ax, run, cost_column, xlabel, title, title_loc="left")

top_axes[2].axis("off")
top_axes[2].legend(
    handles=model_legend_handles(run), title="Model", loc="center left", frameon=False
)
per_case_panel(bottom.subplots(), run, "C) Accuracy by test case", title_loc="left")
export_figure(
    fig,
    settings,
    "planner_quality_cost_per_case",
    {"quality_cost": trade_off_data(run), "per_case": per_case_data(run)},
)
plt.show()
plt.close(fig)